### Adzuna API manual data collection

In [1]:
#install packages
!pip install requests pandas

Defaulting to user installation because normal site-packages is not writeable


In [11]:
import os
import math
import time
import glob
import requests
import pandas as pd
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

In [12]:
# Adzuna API Credentials
APP_ID = "ff1db3d5"
APP_KEY = "8f82686b308b995e71896278c8ec5884"
jobs_per_page = 100


In [13]:
# Create Output Folder

output_folder = "data/raw/adzuna"
os.makedirs(output_folder, exist_ok=True)


In [14]:
# First Request

url = "https://api.adzuna.com/v1/api/jobs/gb/search/1/"

params = {
    "app_id": APP_ID,
    "app_key": APP_KEY,
    "results_per_page": jobs_per_page
}

In [15]:
response = requests.get(url, params=params)
response.raise_for_status()

data = response.json()

total_jobs = data["count"]
total_pages = math.ceil(total_jobs / jobs_per_page)

print(f"Total Jobs : {total_jobs}")
print(f"Total Pages: {total_pages}")

Total Jobs : 729307
Total Pages: 7294


In [16]:
# Retry Session
session = requests.Session()

retry_strategy = Retry(
    total=5,
    backoff_factor=2,
    status_forcelist=[429, 500, 502, 503, 504]
)

adapter = HTTPAdapter(max_retries=retry_strategy)

session.mount("https://", adapter)
session.mount("http://", adapter)

In [ ]:
#automatic resume system after the daily limit

all_jobs = []
seen_ids = set()

pages_per_csv = 1000

existing_files = sorted(
    glob.glob(os.path.join(output_folder, "jobs_data_part_*.csv"))
)

completed_parts = len(existing_files)

csv_counter = completed_parts + 1
start_page = completed_parts * pages_per_csv + 1

In [18]:
# Download Pages
#code for all pages (reached the adzuna api daily limit)
#for page in range(1, total_pages + 1):

#automatic resume system
for page in range(start_page, total_pages + 1):

#bcz for daily limit i need to re-download from page no. 5001 (2line of code)
#start_page = 5001

#for page in range(start_page, total_pages + 1):

    page_url = f"https://api.adzuna.com/v1/api/jobs/gb/search/{page}/"

    try:

        response = session.get(
            page_url,
            params=params,
            timeout=30
        )

        response.raise_for_status()

        page_data = response.json()

        results = page_data.get("results", [])

        print(
            f"Page {page}/{total_pages} "
            f"Retrieved {len(results)} jobs"
        )

        for job in results:

            job_id = str(job["id"])

            if job_id not in seen_ids:
                seen_ids.add(job_id)
                all_jobs.append(job)

    except Exception as e:

        print(f"Error on page {page}: {e}")
        continue

    # Save every 1000 pages

    if page % pages_per_csv == 0 or page == total_pages:

        df = pd.DataFrame(all_jobs)

        filename = os.path.join(
            output_folder,
            f"jobs_data_part_{csv_counter}.csv"
        )

        df.to_csv(filename, index=False)

        print(f"\nSaved {filename}")
        print(f"Rows Saved : {len(df)}")
        print(f"Unique IDs : {df['id'].nunique()}\n")

        all_jobs = []
        csv_counter += 1

    time.sleep(1)

print("\nDownload Complete!")


Page 5001/7294 Retrieved 50 jobs
Page 5002/7294 Retrieved 50 jobs
Page 5003/7294 Retrieved 50 jobs
Page 5004/7294 Retrieved 50 jobs
Page 5005/7294 Retrieved 50 jobs
Page 5006/7294 Retrieved 50 jobs
Page 5007/7294 Retrieved 50 jobs
Page 5008/7294 Retrieved 50 jobs
Page 5009/7294 Retrieved 50 jobs
Page 5010/7294 Retrieved 50 jobs
Page 5011/7294 Retrieved 50 jobs
Page 5012/7294 Retrieved 50 jobs
Page 5013/7294 Retrieved 50 jobs
Page 5014/7294 Retrieved 50 jobs
Page 5015/7294 Retrieved 50 jobs
Page 5016/7294 Retrieved 50 jobs
Page 5017/7294 Retrieved 50 jobs
Page 5018/7294 Retrieved 50 jobs
Page 5019/7294 Retrieved 50 jobs
Page 5020/7294 Retrieved 50 jobs
Page 5021/7294 Retrieved 50 jobs
Page 5022/7294 Retrieved 50 jobs
Page 5023/7294 Retrieved 50 jobs
Page 5024/7294 Retrieved 50 jobs
Page 5025/7294 Retrieved 50 jobs
Page 5026/7294 Retrieved 50 jobs
Page 5027/7294 Retrieved 50 jobs
Page 5028/7294 Retrieved 50 jobs
Page 5029/7294 Retrieved 50 jobs
Page 5030/7294 Retrieved 50 jobs
Page 5031/

In [19]:
# Merge CSV Files
print("\nMerging CSV files...")

files = sorted(
    glob.glob(
        os.path.join(
            output_folder,
            "jobs_data_part_*.csv"
        )
    )
)

master_df = pd.concat(
    [pd.read_csv(f) for f in files],
    ignore_index=True
)

print("\nBefore Removing Duplicates")
print("Rows       :", len(master_df))
print("Unique IDs :", master_df["id"].nunique())

master_df = master_df.drop_duplicates(
    subset="id"
)

master_file = os.path.join(
    output_folder,
    "adzuna_jobs.csv"
)

master_df.to_csv(
    master_file,
    index=False
)

print("\nAfter Removing Duplicates")
print("Rows       :", len(master_df))
print("Unique IDs :", master_df["id"].nunique())

print(f"\nMaster file saved to:\n{master_file}")


Merging CSV files...

Before Removing Duplicates
Rows       : 6279
Unique IDs : 6275

After Removing Duplicates
Rows       : 6275
Unique IDs : 6275

Master file saved to:
data/raw/adzuna\adzuna_jobs.csv


In [20]:
print(df.columns.tolist())
print(df.shape)
df.head()

['longitude', 'category', '__CLASS__', 'salary_is_predicted', 'adref', 'contract_type', 'created', 'location', 'latitude', 'title', 'description', 'company', 'contract_time', 'salary_min', 'id', 'redirect_url', 'salary_max']
(50, 17)


,longitude,category,__CLASS__,salary_is_predicted,adref,contract_type,created,location,latitude,title,description,company,contract_time,salary_min,id,redirect_url,salary_max
0,-2.165740,"{'label': 'Domestic help & Cleaning Jobs', '__...",Adzuna::API::Response::Job,0,eyJhbGciOiJIUzI1NiJ9.eyJpIjoiNTgwNTg1MjcxNCIsI...,permanent,2026-07-18T06:36:31Z,"{'area': ['UK', 'West Midlands', 'Kingswinford...",52.498100,Experienced End of Tenancy (EOT) Cleaners,A UK cleaning company is recruiting experience...,"{'display_name': 'Emplyfy', '__CLASS__': 'Adzu...",part_time,31200.00,5805852714,https://www.adzuna.co.uk/jobs/details/58058527...,31200.00
1,-2.941920,{'__CLASS__': 'Adzuna::API::Response::Category...,Adzuna::API::Response::Job,1,eyJhbGciOiJIUzI1NiJ9.eyJpIjoiNDQwNDc3NjMzNCIsI...,NaN,2023-10-31T19:36:12Z,{'__CLASS__': 'Adzuna::API::Response::Location...,53.485500,Online Music Tutoring Job in Aintree – Flexibl...,Become an online tutor in Music and support st...,{'__CLASS__': 'Adzuna::API::Response::Company'...,part_time,44110.04,4404776334,https://www.adzuna.co.uk/jobs/details/44047763...,44110.04
2,-3.967750,{'__CLASS__': 'Adzuna::API::Response::Category...,Adzuna::API::Response::Job,1,eyJhbGciOiJIUzI1NiJ9.eyJzIjoiS3J0R2hlU0Y4UkdiR...,NaN,2025-12-01T16:38:52Z,"{'display_name': 'Cambusbarron, Stirling', 'ar...",56.108898,Become a Chinese Tutor in Cambusbarron – Flexi...,Help students in Cambusbarron succeed in Chine...,{'__CLASS__': 'Adzuna::API::Response::Company'...,NaN,43371.71,5521941693,https://www.adzuna.co.uk/jobs/details/55219416...,43371.71
3,-1.149170,{'__CLASS__': 'Adzuna::API::Response::Category...,Adzuna::API::Response::Job,1,eyJhbGciOiJIUzI1NiJ9.eyJzIjoiS3J0R2hlU0Y4UkdiR...,NaN,2024-09-05T22:25:09Z,{'__CLASS__': 'Adzuna::API::Response::Location...,52.955101,Music Teacher Position – Immediate! Start in N...,Music Teacher Position – Immediate Start in No...,{'__CLASS__': 'Adzuna::API::Response::Company'...,NaN,32246.56,4853642777,https://www.adzuna.co.uk/jobs/details/48536427...,32246.56
4,0.131237,"{'tag': 'scientific-qa-jobs', '__CLASS__': 'Ad...",Adzuna::API::Response::Job,1,eyJhbGciOiJIUzI1NiJ9.eyJpIjoiNTc5NzQwOTgwOSIsI...,NaN,2026-07-12T00:08:23Z,{'__CLASS__': 'Adzuna::API::Response::Location...,52.202499,Postdoctoral Scientist | Neurobiology | Dr Ann...,"£42,694 per annum Overall purpose: To work wit...",{'__CLASS__': 'Adzuna::API::Response::Company'...,NaN,32440.35,5797409809,https://www.adzuna.co.uk/jobs/details/57974098...,32440.35
